## How to Use

1. Run the setup cell to install dependencies.
2. Execute each section sequentially and discuss the outputs with your audience.
3. All models run on CPU by default, so Colab GPU acceleration is optional.

In [ ]:
!pip install -q transformers accelerate sentence-transformers gradio pillow rich datasets

In [ ]:
from io import BytesIO
from pathlib import Path

import requests
from PIL import Image, ImageDraw, ImageFont
import torch
from IPython.display import display

from transformers import BlipForQuestionAnswering, BlipProcessor
from transformers import DonutProcessor, VisionEncoderDecoderModel
from sentence_transformers import SentenceTransformer

torch.set_grad_enabled(False)
DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

## Visual Question Answering with BLIP

We will load an open-source BLIP VQA model and ask it questions about a Creative Commons image (sourced from Unsplash).

In [ ]:
BLIP_MODEL_ID = "Salesforce/blip-vqa-base"
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_ID)
blip_model = BlipForQuestionAnswering.from_pretrained(BLIP_MODEL_ID)
blip_model.to(DEVICE)
blip_model.eval()
print("BLIP model ready.")

In [ ]:
street_image_url = "https://images.unsplash.com/photo-1520975928316-040be9c3a08b?auto=format&fit=crop&w=1200&q=80"
response = requests.get(street_image_url, timeout=20)
response.raise_for_status()
vqa_image = Image.open(BytesIO(response.content)).convert("RGB")
display(vqa_image)

question = "How many cyclists are in the scene?"
inputs = blip_processor(images=vqa_image, text=question, return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
with torch.no_grad():
    output_ids = blip_model.generate(**inputs, max_new_tokens=32)
answer = blip_processor.decode(output_ids[0], skip_special_tokens=True).strip()
print(f"Q: {question}")
print(f"A: {answer}")

## Document Question Answering with Donut

Next we generate a synthetic invoice image and query it with the Donut DocVQA model, which reads documents without OCR.

In [ ]:
DONUT_MODEL_ID = "naver-clova-ix/donut-base-finetuned-docvqa"
doc_processor = DonutProcessor.from_pretrained(DONUT_MODEL_ID)
doc_model = VisionEncoderDecoderModel.from_pretrained(DONUT_MODEL_ID)
doc_model.to(DEVICE)
doc_model.eval()
print("Donut model ready.")

In [ ]:
invoice_image = Image.new("RGB", (900, 600), "white")
draw = ImageDraw.Draw(invoice_image)
font = ImageFont.load_default()
lines = [
    "AURORA SUPPLIES",
    "Invoice #: 2025-113",
    "Bill To: Horizon Labs",
    "Item               Qty    Unit    Total",
    "Safety goggles      10    15.00   150.00",
    "Lab coats            6    32.00   192.00",
    "Gloves (box)        12     6.50    78.00",
    "----------------------------------------",
    "Subtotal:                          420.00",
    "Tax (5%):                          21.00",
    "Invoice Total:                     441.00",
    "Due Date: 22-Dec-2025",
]
y = 40
for line in lines:
    draw.text((40, y), line, fill="black", font=font)
    y += 40

display(invoice_image)

doc_question = "What is the invoice total?"
pixel_values = doc_processor(invoice_image, return_tensors="pt")
pixel_values = {k: v.to(DEVICE) for k, v in pixel_values.items()}
prompt = f"<s_docvqa><s_question>{doc_question}</s_question><s_answer>"
decoder_input_ids = doc_processor.tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
)["input_ids"].to(DEVICE)
with torch.no_grad():
    output_ids = doc_model.generate(
        pixel_values=pixel_values["pixel_values"],
        decoder_input_ids=decoder_input_ids,
        max_length=512,
        num_beams=3,
    )
decoded = doc_processor.batch_decode(output_ids, skip_special_tokens=True)[0]
answer_json = doc_processor.token2json(decoded)
doc_answer = answer_json.get("answer", decoded).strip()
print(f"Q: {doc_question}")
print(f"A: {doc_answer}")

## Multimodal Retrieval with CLIP

Finally we embed a small corpus of domain images and retrieve the best matches for a text query using a multilingual CLIP checkpoint. Image sources: Unsplash (free to use).

In [ ]:
RETRIEVAL_MODEL_ID = "sentence-transformers/clip-ViT-B-32-multilingual-v1"
retrieval_model = SentenceTransformer(RETRIEVAL_MODEL_ID, device=DEVICE.type)
print("Retrieval model ready.")

In [ ]:
asset_dir = Path("retrieval_assets")
asset_dir.mkdir(exist_ok=True)
image_sources = {
    "street.jpg": "https://images.unsplash.com/photo-1520975928316-040be9c3a08b?auto=format&fit=crop&w=800&q=80",
    "construction.jpg": "https://images.unsplash.com/photo-1489515217757-5fd1be406fef?auto=format&fit=crop&w=800&q=80",
    "laboratory.jpg": "https://images.unsplash.com/photo-1581091012184-7f928e417cd4?auto=format&fit=crop&w=800&q=80",
}
captions = {
    "street.jpg": "Cyclists navigating a city street",
    "construction.jpg": "Engineer inspecting a high-rise construction site",
    "laboratory.jpg": "Scientists collaborating in a laboratory",
}
image_paths = []
pil_images = []
for name, url in image_sources.items():
    path = asset_dir / name
    if not path.exists():
        resp = requests.get(url, timeout=20)
        resp.raise_for_status()
        path.write_bytes(resp.content)
    image = Image.open(path).convert("RGB")
    pil_images.append(image)
    image_paths.append(path)
    print(f"Loaded {name} - {captions[name]}")

query = "a construction worker wearing safety gear"
print(f"\nQuery: {query}")
image_embeddings = retrieval_model.encode(pil_images, convert_to_tensor=True, device=DEVICE.type, show_progress_bar=False)
image_embeddings = torch.nn.functional.normalize(image_embeddings, p=2, dim=1)
query_embedding = retrieval_model.encode(query, convert_to_tensor=True, device=DEVICE.type, show_progress_bar=False)
query_embedding = torch.nn.functional.normalize(query_embedding, p=2, dim=0)
scores = image_embeddings @ query_embedding
topk = torch.topk(scores, k=len(image_paths))

for rank, idx in enumerate(topk.indices.tolist(), start=1):
    print(f"Rank {rank}: {image_paths[idx].name} (score={scores[idx].item():.3f}) - {captions[image_paths[idx].name]}")
    display(pil_images[idx])

## Wrap-Up

- Swap the sample assets or questions to align with your audience's domain.
- Each section runs on CPU, so you can reproduce the seminar live on Colab or a standard laptop.
- Combine these building blocks with retrieval-augmented generation or lightweight fine-tuning for cross-domain pilots.